# How to extract data from NWB



### Import modules

In [1]:
from pynwb import NWBHDF5IO
from nwbwidgets import nwb2widget
import ndx_pose
import numpy as np
import matplotlib.pyplot as plt
from importlib import sys, reload

ImportError: cannot import name 'call_docval_func' from 'hdmf.utils' (/opt/anaconda3/envs/HatLab/lib/python3.13/site-packages/hdmf/utils.py)

### Define nwbfile path and open it in read mode

In [ ]:
nwb_processed_file = '/project/nicho/data/marmosets/electrophys_data_for_processing/TYTR20250216_0830_staticAndStaticFree/TYTR20250216_0830_staticAndStaticFree001_processed.nwb'
io_prc = NWBHDF5IO(nwb_processed_file, mode='r')
nwb_prc = io_prc.read()

In [ ]:
nwb_acquisition_file = '/project/nicho/data/marmosets/electrophys_data_for_processing/TYTR20250216_0830_staticAndStaticFree/TYTR20250216_0830_staticAndStaticFree001_acquisition.nwb'
io_acq = NWBHDF5IO(nwb_acquisition_file, mode='r')
nwb_acq = io_acq.read()

### Use nwb2widget to explore the data

In [ ]:
nwb2widget(nwb_prc)


In [ ]:
nwb2widget(nwb_acq)

## Accessing Relevant Information

### Get Trial Periods
Because task is unrestrained, marmosets choose to voluntarily start trial blocks. Each block is a different video ("event"), so within each video there could be multiple trials. We can get the start and stop time of each trial using the kinematics of the arm. 

In [26]:
# get each video event time intervals, in seconds
np.array(nwb_prc.intervals['video_events_static']) # len = 11, so 11 events
np.array(nwb_prc.intervals['video_events_static'].start_time)
np.array(nwb_prc.intervals['video_events_static'].stop_time)

# get each reach trial time stamp, in seconds
np.array(nwb_prc.intervals['reaching_segments_static']) # len = 84, so 84 trials
np.array(nwb_prc.intervals['reaching_segments_static'].start_time)
np.array(nwb_prc.intervals['reaching_segments_static'].stop_time)

array([ 462.68669985,  475.11393317,  480.48083317, 1000.36519941,
       1005.64543274, 1010.47233273, 1014.6324994 , 1022.48619939,
       1025.32633272, 1028.79313272, 1038.87359938, 1045.36719937,
       1052.68086603, 1062.96799936, 1541.93459897, 1549.40826563,
       1556.06856562, 1567.48239894, 1572.44929894, 1579.08289894,
       1596.63703225, 1611.61769891, 1625.77163223, 1637.35883222,
       1641.72569888, 1915.18749866, 1939.98193197, 1944.10209864,
       1952.3224653 , 2228.89446507, 2241.54169839, 2263.63603171,
       2269.43626504, 2279.57673169, 2286.65036502, 2289.57716502,
       2293.98403168, 2333.78579832, 2352.3065983 , 2375.53429828,
       2384.10799828, 2554.57703147, 2578.08476478, 2593.68543144,
       3388.57423079, 3392.54776412, 3395.84789745, 3474.07803072,
       3490.77209737, 3494.7589307 , 3516.29989735, 3522.98016401,
       3526.03363067, 3553.50819732, 3601.84363061, 3606.47049727,
       3639.31196391, 3779.12473047, 4072.67616356, 4087.82349

### Get Kinematic Trajectories
kinematics are 2d, measured as extension from the body. this animal is left handed, so we track the extension using the l-wrist marker from DLC. this shows how to get the kinematics of each trial as well as the timestamps of those positions

In [16]:
# example for one trial
trial = 0
segment_df = nwb_prc.intervals['reaching_segments_static'].to_dataframe()
segment_info = segment_df.iloc[trial]
wrist_position = nwb_prc.processing[segment_info.kinematics_module].data_interfaces[segment_info.video_event].pose_estimation_series['l-wrist'].data[:]
wrist_position_timestamps = nwb_prc.processing[segment_info.kinematics_module].data_interfaces[segment_info.video_event].pose_estimation_series['l-wrist'].timestamps[:]

### Get Neural Data
Data is spike sorted using ironclust. Individual units are traced back to the original electrode label on the map file. Recall the array is on the left on this animal, so the wire bundle extends towards the medial wall and the back. so on the map file, the bottom row is the most anterior column, and the right column marked with wire bundle is the medial wall. Array has 96 channels, also collected raw from 3 analog inputs for a total of 99 entries in raw electrical series.

In [38]:
# access spike information of one example unit
unit = 0
nwb_prc.units[unit]

# find location on array - this matches the electrode label on the map file
elec_label = nwb_prc.units[unit].electrode_label 
idx = np.where(elec_label[unit] == nwb_prc.electrodes[:].electrode_label)[0]
print(nwb_prc.electrodes[idx].x, nwb_prc.electrodes[idx].y, nwb_prc.electrodes[idx].z) # get [x, y, z] of channel (all 1mm length)

# downsample to only trial periods
idx = 0
units = nwb_prc.units.to_dataframe()
spikes = units.spike_times.iloc[idx] # where idx = idx of unit in units
event_spikes = [spike for spike in spikes if segment_info.start_time<spike<segment_info.stop_time]

id
2    3200.0
Name: x, dtype: float64 id
2    1600.0
Name: y, dtype: float64 id
2   -1000.0
Name: z, dtype: float64


In [ ]:
# access lfp information of one example unit
channel = 0  # channel id, not electrode_label
nwb_acq.acquisition['ElectricalSeries']
lfp = nwb_acq.acquisition['ElectricalSeries'].data[:, channel]
# find channel on the array
nwb_acq.acquisition['ElectricalSeries'].electrodes.table[channel].electrode_label 

### When you finish working with the data, close the files

In [ ]:
io_acq.close()

In [ ]:
io_prc.close()